### SETUP

In [21]:
%pip install langchain_community langchainhub chromadb langchain langchain-groq

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
%pip install python-dotenv
from dotenv import load_dotenv
import os

load_dotenv()
GROQ_API_KEY=os.getenv("GROQ_API_KEY")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
import os
from langchain_community.document_loaders import WebBaseLoader
os.environ["USER_AGENT"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
loader =WebBaseLoader(web_paths=["https://www.educosys.com/course/genai"])
docs=loader.load()
print(docs)

[Document(metadata={'source': 'https://www.educosys.com/course/genai', 'language': 'en'}, page_content='')]


In [56]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Load the locally saved course text
loader = TextLoader("../data/genai_course.txt", encoding="utf-8")
docs = loader.load()

print(f"Successfully loaded {len(docs[0].page_content)} characters.")

# 2. Split the text
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200
)
course_splits = text_splitter.split_documents(docs)

# 3. Append to your existing Chroma vector store
vectorstore.add_documents(documents=course_splits)
print(f"Successfully added {len(course_splits)} course chunks to the vector store.")

Successfully loaded 4815 characters.
Successfully added 6 course chunks to the vector store.


In [58]:
print(splits[0].page_content)


Test Mode

Application error: a server-side exception has occurred while loading educosys.com (see the server logs for more information).

Digest: 2980155944


In [50]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(
    documents=splits, 
    embedding=embedding_model
)


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2692.09it/s]


In [27]:
retriever=vectorstore.as_retriever()

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.

Question: {question} 

Context: {context} 

Answer:"""

prompt = PromptTemplate.from_template(template)
print(prompt)


# Use above code instead of below one
# import langchainhub as hub

# prompt = hub.pull("rlm/rag-prompt")
# print(prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\n\nQuestion: {question} \n\nContext: {context} \n\nAnswer:"


In [31]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [30]:
def format_docs(docs):
    return"\n".join(doc.page_content for doc in docs)

In [41]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

# You can replace "llama3-8b-8192" with whichever Groq model you prefer
llm = ChatGroq(model_name="openai/gpt-oss-20b")

# 2. Build the chain with the LLM at the end
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [60]:
rag_chain.invoke("Are certificates provided?")

'Yes, certificates are provided. A verifiable digital certificate is issued upon successful completion of all modules, coding exercises, and the final capstone project. The certificate includes a unique credential ID, a detailed transcript of skills acquired, and a link to your deployed capstone project.'